# Práctica guiada: clasificación con Regresión Logística
## Introducción a la Regresión Logística 



**Objetivos:**

1. Explicar qué es un problema de **clasificación** y en qué se diferencia de uno de regresión.
2. Entender qué es la **función sigmoide** y cómo la regresión logística la usa para calcular **probabilidades**.
3. Crear, entrenar y usar un modelo de regresión logística con **Scikit-Learn**.
4. Diferenciar entre **predecir una clase** (`predict`) y **predecir una probabilidad** (`predict_proba`).
5. Evaluar un clasificador con **accuracy, matriz de confusión, precision, recall y F1**, y saber cuándo conviene cada una.
6. Visualizar la **frontera de decisión** del modelo con uno y con dos atributos.
7. Conocer las variantes (multiclase, regularización), las ventajas, las desventajas y los límites del método.

---


>  **Repaso rápido de lo anterior.** En la regresión lineal el modelo predecía un **número** (un precio, una temperatura). Acá el modelo va a predecir una **categoría** (¿spam o no spam?, ¿aprueba o no aprueba?). Vas a reconocer casi todo el flujo de trabajo: instanciar → entrenar (`fit`) → predecir (`predict`) → evaluar.

---
# 1. Conceptos previos: ¿qué es clasificar?

## 1.1 Regresión vs. clasificación

Dentro del **aprendizaje supervisado** (donde entrenamos con datos que incluyen la respuesta correcta) hay dos grandes familias, según lo que queremos predecir:

| Problema | ¿Qué predice? | Ejemplos |
|---|---|---|
| **Regresión** | Un **número continuo** | Precio de una casa, temperatura de mañana |
| **Clasificación** | Una **categoría** (clase) | ¿Spam o no spam? ¿Tumor benigno o maligno? ¿Aprobará o no? |

**La regresión logística es un algoritmo de *clasificación***. Que no te confunda el nombre: aunque diga "regresión", **no predice números sino categorías**.

## 1.2 Clasificación binaria y multiclase

- **Clasificación binaria:** hay **solo dos** categorías posibles. Por convención se las llama **clase 0** y **clase 1**.
- **Clasificación multiclase:** hay **tres o más** categorías (por ejemplo, reconocer si una foto es de un perro, un gato o un pájaro).

| Ejemplo | Clase 0 (negativa) | Clase 1 (positiva) |
|---|---|---|
| Filtro de spam | Correo legítimo | Spam |
| Test médico | Persona sana | Persona enferma |
| Test de embarazo | Negativo | Positivo |
| Préstamo bancario | Devuelve el préstamo | No lo devuelve |

>  **"Positivo" no significa "bueno".** La clase positiva (1) es simplemente **la que nos interesa detectar**, sea buena o mala. En un test médico, "positivo" significa que la enfermedad *fue detectada*.

En esta práctica nos concentramos en la **clasificación binaria** y, hacia el final, vemos cómo se extiende a más clases.

## 1.3 Vocabulario nuevo

Además del vocabulario de la práctica anterior (atributo, etiqueta, entrenar, predecir...), vas a ver estos términos:

| Término | Significado | Ejemplo (filtro de spam) |
|---|---|---|
| **Clase** | Cada una de las categorías posibles | "spam" y "no spam" |
| **Probabilidad** | Número entre 0 y 1 que indica qué tan probable es algo (0 = imposible, 1 = seguro) | 0,92 = "92 % de probabilidad de que sea spam" |
| **Umbral (*threshold*)** | Valor de probabilidad a partir del cual decidimos que es clase 1 (normalmente 0,5) | Si P(spam) ≥ 0,5 → lo marcamos como spam |
| **Frontera de decisión** | El "borde" que separa la zona donde el modelo predice una clase de la zona donde predice la otra | El valor de `x` donde la probabilidad cruza 0,5 |

---
# 2. ¿Qué es la regresión logística?

## 2.1 La idea

La regresión logística es un modelo **lineal** para **clasificación binaria**. En lugar de predecir directamente "clase 0" o "clase 1", hace dos pasos:

1. **Calcula una probabilidad** de que el registro pertenezca a la clase 1 (un número entre 0 y 1).
2. **Aplica un umbral** (normalmente 0,5) para decidir:
   - si la probabilidad es **mayor o igual que 0,5** → predice **clase 1**;
   - si es **menor que 0,5** → predice **clase 0**.

> **Ejemplo.** Un filtro de spam recibe un correo y calcula "hay un 92 % de probabilidad de que sea spam". Como 0,92 ≥ 0,5, lo clasifica como spam. Para otro correo calcula 0,07: lo clasifica como legítimo.

## 2.2 ¿Por qué no usamos simplemente regresión lineal?

Si hacemos una regresión lineal sobre datos con clases 0 y 1, la recta puede dar resultados como **-0,76 o 2,31**, que **no tienen sentido como probabilidades** (una probabilidad nunca puede ser negativa ni mayor que 1). Además, la recta se ve muy afectada por los valores extremos. Lo vamos a comprobar con código en la sección 4.

Necesitamos una función que **tome cualquier número y lo "aplaste" para que quede siempre entre 0 y 1**. Esa función existe y se llama **sigmoide**.

## 2.3 La función sigmoide

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

donde $e$ es el número de Euler (≈ 2,718), una constante matemática muy usada. No hace falta que recuerdes sus propiedades: lo importante es **cómo se comporta** la función.

| Si `z` es... | `σ(z)` vale aproximadamente... | Interpretación |
|---|---|---|
| Un número **muy negativo** (-10) | cerca de **0** | "Casi seguro que es clase 0" |
| **0** | exactamente **0,5** | "Estoy indeciso" |
| Un número **muy positivo** (+10) | cerca de **1** | "Casi seguro que es clase 1" |

Tiene forma de **"S" suave**: sube lentamente, luego rápido en la zona central y vuelve a aplanarse. **Siempre devuelve un valor entre 0 y 1**, justo lo que necesitamos para una probabilidad.

## 2.4 La ecuación completa de la regresión logística

La entrada de la sigmoide, `z`, es **una función lineal de los atributos** (¡igual que en la regresión lineal!):

$$ z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p \qquad\Longrightarrow\qquad P(\text{clase 1}) = \frac{1}{1 + e^{-z}} $$

Con **un solo atributo**, queda así:

$$ P(\text{clase 1}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}} $$

Los parámetros que el modelo **aprende** tienen un rol parecido al de la pendiente y la ordenada de la regresión lineal:

| Parámetro | Nombre | Qué controla |
|---|---|---|
| **β₁** (o `w1`) | Coeficiente | La **inclinación** de la curva en la zona de cambio. Si es grande, la transición de 0 a 1 es abrupta; si es chico, es gradual. Si es **negativo**, la curva se invierte (baja en lugar de subir) |
| **β₀** (o `w0`) | Intercepto | **Desplaza la curva** hacia la izquierda o la derecha |

> En la documentación y otros materiales vas a ver los parámetros con distintos nombres (`w0` y `w1`, o `β₀` y `β₁`). **Son lo mismo.**

## 2.5 ¿Y por qué se llama "regresión"? (el logit, opcional)

La sigmoide no es una recta, así que ¿cómo se calcula? Con un truco matemático: si invertimos la sigmoide, lo que queda **sí es lineal**:

$$ \ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p $$

- $\dfrac{p}{1-p}$ se llama ***odds*** (en castellano, *razón de probabilidades* o "chances"). Si `p = 0,8`, los odds son `0,8 / 0,2 = 4`: "4 a 1 a favor".
- $\ln\left(\dfrac{p}{1-p}\right)$ se llama **logit**.

Es decir: **el logaritmo de los odds es una función lineal de los atributos**. De ahí viene el "regresión" del nombre: por dentro hay una regresión lineal sobre el logit. En la práctica, Scikit-Learn hace todo esto por vos.

## 2.6 Regresión lineal vs. regresión logística

| | **Regresión lineal** | **Regresión logística** |
|---|---|---|
| **Tipo de problema** | Regresión | Clasificación |
| **¿Qué predice?** | Un número cualquiera | Una probabilidad (0 a 1) → una clase |
| **Función** | Recta: $y = mx + b$ | Sigmoide: $\frac{1}{1+e^{-(\beta_0 + \beta_1 x)}}$ |
| **Qué busca** | La recta que **pasa lo más cerca posible** de los puntos | La curva/frontera que **mejor separa** las clases |
| **Métricas típicas** | MAE, MSE, RMSE, R² | Accuracy, precision, recall, F1 |
| **Clase en Scikit-Learn** | `LinearRegression` | `LogisticRegression` |

## 2.7 ¿Para qué sirve?

Se usa muchísimo cuando hay que **estimar la probabilidad de que algo ocurra**:

-  Filtros de **spam**.
-  **Diagnóstico médico**: probabilidad de que una persona tenga una enfermedad según sus estudios.
-  **Riesgo crediticio**: probabilidad de que alguien no devuelva un préstamo.
-  **Abandono de clientes** (*churn*): probabilidad de que un cliente se dé de baja.
-  **Rendimiento académico**: probabilidad de aprobar según horas de estudio y asistencia.

---
# 3. Preparación: las herramientas que vamos a usar

Son las mismas de la práctica anterior (más algunas funciones nuevas de Scikit-Learn que iremos presentando):

| Librería | Alias | ¿Para qué sirve? |
|---|---|---|
| **NumPy** | `np` | Cálculos numéricos y arreglos |
| **Pandas** | `pd` | Tablas de datos (*DataFrames*) |
| **Matplotlib** | `plt` | Gráficos |
| **Seaborn** | `sns` | Gráficos estadísticos más vistosos |
| **Scikit-Learn** | `sklearn` | Modelos de Machine Learning, métricas y datasets de ejemplo |

>  Si alguna librería no está instalada, quitá el `#` de la primera línea de la celda y ejecutala. **Atención:** Scikit-Learn se instala como `scikit-learn`, aunque se importe como `sklearn`. En Google Colab ya vienen instaladas.

In [1]:
# Si te falta alguna librería, quitá el "#" de la línea siguiente y ejecutá la celda:
# %pip install numpy pandas matplotlib seaborn scikit-learn

import numpy as np                 # cálculos numéricos
import pandas as pd                # tablas de datos
import matplotlib.pyplot as plt    # gráficos
import seaborn as sns              # gráficos estadísticos

sns.set_theme(style="whitegrid")   # estilo de gráficos más prolijo

print("¡Librerías importadas correctamente!")

¡Librerías importadas correctamente!


---
# 4. Regresión logística con un solo atributo

Vamos a trabajar con **datos sintéticos** (generados con código), así sabemos cómo se crearon y podemos juzgar si el modelo hace un buen trabajo.

## 4.1 La función sigmoide en código

Antes de clasificar nada, escribamos la sigmoide y veamos cómo se comporta. Definimos una **función** (un bloque de código reutilizable que se crea con `def`) que recibe:

- `x`: el/los valor(es) del atributo,
- `w0`: el intercepto (β₀),
- `w1`: el coeficiente (β₁).

**Qué hace cada línea:**
- `z = w0 + w1 * x`: calcula la parte **lineal** (¡es la misma fórmula de una recta!).
- `np.exp(-z)`: calcula $e^{-z}$ (la exponencial).
- `1 / (1 + np.exp(-z))`: aplica la sigmoide. Notá que es una **composición**: primero una función lineal y después la sigmoide sobre su resultado.
- `return y`: devuelve el resultado.

In [ ]:
def sigmoid_function(x, w0, w1):
    # Calcula la probabilidad de clase 1 para un valor (o arreglo de valores) x
    z = w0 + w1 * x               # parte lineal (como en regresión lineal)
    y = 1 / (1 + np.exp(-z))      # la sigmoide "aplasta" z para que quede entre 0 y 1
    return y

# Probemos con unos valores sueltos (con w0=0 y w1=1)
for z in [-10, -2, 0, 2, 10]:
    print(f"x = {z:>3}  ->  probabilidad = {sigmoid_function(z, 0, 1):.4f}")

Los resultados confirman lo que vimos en la teoría: para valores **muy negativos** la probabilidad es casi **0**, para **0** es exactamente **0,5**, y para valores **muy positivos** es casi **1**.

### ¿Qué hacen `w0` y `w1`?

Dibujemos varias sigmoides con distintos parámetros para **ver** el efecto de cada uno.

**Detalles de código:**
- `np.linspace(-8, 8, 200)`: crea 200 valores equiespaciados entre -8 y 8 para dibujar una curva suave.
- Recorremos una lista de pares `(w0, w1)` con un bucle `for`, y en cada vuelta dibujamos una curva.
- `plt.axhline(0.5, ...)` dibuja una línea horizontal en 0,5 (el umbral de decisión).

In [ ]:
x_plot = np.linspace(-8, 8, 200)

# Distintas combinaciones de parámetros (w0, w1) para comparar
combinaciones = [(0, 1), (0, 3), (0, -1), (-3, 1), (3, 1)]

plt.figure(figsize=(12, 6))
for w0, w1 in combinaciones:
    plt.plot(x_plot, sigmoid_function(x_plot, w0, w1), linewidth=2, label=f"w0 = {w0}, w1 = {w1}")

plt.axhline(0.5, color="gray", linestyle="--", label="Umbral = 0,5")
plt.title("La función sigmoide con distintos parámetros")
plt.xlabel("x")
plt.ylabel("Probabilidad de clase 1")
plt.legend()
plt.show()

**Cómo leer el gráfico (comparando con la curva base `w0 = 0, w1 = 1`):**

- **`w1 = 3`** → la curva es **más empinada**: el modelo "se decide" más rápido entre 0 y 1.
- **`w1 = -1`** → la curva está **invertida**: a mayor `x`, **menor** probabilidad de clase 1.
- **`w0 = -3`** → la curva se **desplaza hacia la derecha**: hace falta un `x` más grande para llegar a 0,5.
- **`w0 = 3`** → la curva se **desplaza hacia la izquierda**.

**Lo que hace la regresión logística es encontrar los valores de `w0` y `w1` que mejor ajustan los datos de cada clase.**

## 4.2 Crear el dataset sintético

Usamos la función `make_blobs` de Scikit-Learn, que genera "nubes" (*blobs*) de puntos, una por cada clase. Sus parámetros:

| Parámetro | Significado | Valor que usamos |
|---|---|---|
| `n_samples` | Cantidad total de registros | 1000 |
| `centers` | Cantidad de nubes (clases) | 2 |
| `n_features` | Cantidad de atributos | 1 |
| `cluster_std` | Dispersión de cada nube (más alto = más superposición entre clases) | 1,5 |
| `random_state` | Semilla para que el resultado sea reproducible | 1 |

Devuelve dos cosas: `X` (los atributos) e `y` (la **clase** de cada registro: 0 o 1).

In [ ]:
from sklearn.datasets import make_blobs      # función para crear datasets sintéticos de clasificación

X, y = make_blobs(n_samples=1000, centers=2, n_features=1,
                  random_state=1, cluster_std=1.5)

print("Forma de X:", X.shape, "-> 1000 filas y 1 columna (1 atributo)")
print("Forma de y:", y.shape, "-> 1000 valores (la clase de cada registro)")
print("\nPrimeros 5 valores de X:", X[:5].ravel().round(2))
print("Primeras 5 clases (y):  ", y[:5])

# ¿Cuántos registros hay de cada clase? (np.bincount cuenta cuántas veces aparece cada valor)
print("\nCantidad de registros por clase [clase 0, clase 1]:", np.bincount(y))

Hay **500 registros de cada clase**: un dataset **balanceado** (más adelante veremos por qué esto importa).

### Graficar los datos

Hacemos dos gráficos:

1. **Dispersión (scatter):** el eje horizontal es `x` y el vertical es la clase (0 o 1). Los puntos de cada clase se ubican en una de las dos "líneas" horizontales. Usamos `c=y` para colorear según la clase y `cmap=cm.coolwarm` para usar una paleta azul-rojo.
2. **Histograma por clase:** cuenta cuántos puntos de cada clase caen en cada rango de `x`. Es más fácil ver **dónde se superponen** las clases.

In [ ]:
from matplotlib import cm     # paletas de colores

plt.figure(figsize=(14, 5))

# --- Gráfico 1: dispersión (x vs clase) ---
plt.subplot(1, 2, 1)
plt.scatter(X, y, c=y, alpha=0.5, lw=0.5, edgecolor="grey", cmap=cm.coolwarm)
plt.yticks([0, 1])                       # mostramos solo 0 y 1 en el eje vertical
plt.title("Dispersión: x vs. clase")
plt.xlabel("x")
plt.ylabel("Clase")

# --- Gráfico 2: histograma de x separado por clase ---
plt.subplot(1, 2, 2)
sns.histplot(x=X.ravel(), hue=y, bins=30, palette=["tab:blue", "tab:red"], alpha=0.6)
plt.title("Distribución de x en cada clase")
plt.xlabel("x")

plt.tight_layout()
plt.show()

Se ve claramente que:

- La **clase 0 (azul)** se concentra en valores **bajos** de `x`.
- La **clase 1 (roja)** se concentra en valores **altos** de `x`.
- Hay una **zona intermedia donde las clases se mezclan**. Allí es donde el modelo tendrá más dificultad (y donde cometerá la mayoría de sus errores).

## 4.3 ¿Por qué no usar regresión lineal para clasificar?

Probemos lo que dijimos en la teoría: ajustar una **regresión lineal** a estos datos (tratando 0 y 1 como si fueran números) y ver qué predice en valores de `x` bajos y altos.

In [ ]:
from sklearn.linear_model import LinearRegression

lineal = LinearRegression().fit(X, y)

# Probamos con un x muy bajo (-8) y uno muy alto (15)
for x_prueba in [-8, 15]:
    pred = lineal.predict([[x_prueba]])[0]
    print(f"Para x = {x_prueba:>3} la regresión lineal 'predice' {pred:.2f}")

# Gráfico
x_linea = np.linspace(-8, 14, 100).reshape(-1, 1)
plt.figure(figsize=(12, 5))
plt.scatter(X, y, c=y, alpha=0.4, edgecolor="grey", lw=0.5, cmap=cm.coolwarm)
plt.plot(x_linea, lineal.predict(x_linea), color="green", linewidth=2, label="Regresión lineal")
plt.axhline(0, color="gray", linestyle=":")
plt.axhline(1, color="gray", linestyle=":")
plt.title("Una recta no sirve como probabilidad: se sale del rango [0, 1]")
plt.xlabel("x")
plt.ylabel("Clase / valor predicho")
plt.legend()
plt.show()

La recta verde **se pasa por debajo de 0 y por encima de 1**: para `x = -8` "predice" un número negativo y para `x = 15` un número mayor que 1. **Una probabilidad no puede ser negativa ni mayor que 1**, por eso la regresión lineal no sirve para esto. La sigmoide resuelve el problema: **siempre** devuelve valores entre 0 y 1.

## 4.4 Dividir en entrenamiento y prueba

Igual que en la regresión lineal, **no evaluamos al modelo con los mismos datos con los que aprendió**: separamos un conjunto de **entrenamiento** (para aprender) y uno de **prueba** (para evaluar con datos "nuevos").

Hay una novedad: el parámetro **`stratify=y`**. Significa **estratificar**: la división respeta la **proporción de cada clase** en ambos conjuntos. Si el dataset tiene 50 % de clase 0 y 50 % de clase 1, tanto el conjunto de entrenamiento como el de prueba tendrán esa misma proporción. Es especialmente importante cuando las clases están desbalanceadas, porque evita que, por mala suerte, una clase casi no aparezca en el conjunto de prueba.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,       # 30% para prueba
    random_state=42,      # sorteo reproducible
    stratify=y            # respeta la proporción de clases en ambos conjuntos
)

print("Entrenamiento:", X_train.shape[0], "registros -> por clase:", np.bincount(y_train))
print("Prueba:       ", X_test.shape[0], "registros -> por clase:", np.bincount(y_test))

## 4.5 Entrenar el modelo

Aplicamos el mismo flujo de tres pasos de Scikit-Learn:

| Paso | Código |
|---|---|
| **1. Instanciar** | `clf = LogisticRegression()` |
| **2. Entrenar** | `clf.fit(X_train, y_train)` |
| **3. Predecir** | `clf.predict(X_test)` |

> 📝 Por convención, a los modelos de clasificación se los suele llamar `clf` (de *classifier*, "clasificador"), en lugar de `modelo`. Es solo un nombre de variable: podés usar el que quieras.

Después de entrenar, el modelo guarda los parámetros aprendidos (los que terminan en `_`):

- `clf.intercept_` → el intercepto **β₀** (`w0`).
- `clf.coef_` → los coeficientes **β₁, β₂...** (`w1`, ...).

**Por qué `clf.coef_[0][0]`:** en clasificación, `coef_` es una **tabla de dos dimensiones** (una fila por clase "calculada", una columna por atributo). En la clasificación binaria hay una sola fila y, con un atributo, una sola columna. Entonces `coef_[0][0]` es "fila 0, columna 0": el único coeficiente. En cambio, `intercept_` es un arreglo de una dimensión, por eso se accede con `intercept_[0]`.

In [ ]:
from sklearn.linear_model import LogisticRegression    # la clase del algoritmo

clf = LogisticRegression()          # 1. instanciar
clf.fit(X_train, y_train)           # 2. entrenar (solo con datos de entrenamiento)

w0 = clf.intercept_[0]              # intercepto aprendido
w1 = clf.coef_[0][0]                # coeficiente aprendido

print("Forma de coef_:     ", clf.coef_.shape, "(1 fila x 1 columna)")
print(f"Intercepto (w0):    {w0:.3f}")
print(f"Coeficiente (w1):   {w1:.3f}")
print(f"\nEcuación aprendida: P(clase 1) = 1 / (1 + e^-({w0:.3f} + {w1:.3f} * x))")

El coeficiente `w1` es **positivo**: a mayor `x`, mayor probabilidad de ser clase 1 (coincide con lo que vimos en el histograma). El valor de `w0` es negativo, lo que **desplaza la curva a la derecha**: hace falta un `x` suficientemente grande para llegar al 50 %.

## 4.6 Predecir: clases vs. probabilidades

Este es **el concepto más importante** que diferencia a un clasificador de un modelo de regresión. El modelo ofrece **dos formas** de predecir:

| Método | ¿Qué devuelve? | Ejemplo |
|---|---|---|
| **`clf.predict(X)`** | La **clase** predicha (0 o 1) | `1` |
| **`clf.predict_proba(X)`** | La **probabilidad de cada clase** (una columna por clase) | `[0.08, 0.92]` → 8 % clase 0, 92 % clase 1 |

Las dos columnas de `predict_proba` **siempre suman 1**. La primera es P(clase 0) y la segunda, P(clase 1). Con la clase por defecto, `predict` simplemente elige **la clase con mayor probabilidad** (es decir, aplica el umbral de 0,5).

Armemos una tabla con los primeros 8 registros de prueba para ver las tres cosas juntas: el valor real, la probabilidad y la clase predicha.

In [ ]:
y_test_pred = clf.predict(X_test)               # clases predichas (0 o 1)
y_test_proba = clf.predict_proba(X_test)        # probabilidades: columna 0 = P(clase 0), columna 1 = P(clase 1)

tabla = pd.DataFrame({
    "x": X_test.ravel(),
    "P(clase 0)": y_test_proba[:, 0],
    "P(clase 1)": y_test_proba[:, 1],
    "Clase predicha": y_test_pred,
    "Clase real": y_test,
})
tabla.head(8).round(3)

Observá cómo funciona la decisión: cuando `P(clase 1)` supera 0,5, la clase predicha es 1; cuando no, es 0. Y fijate que **hay casos donde la predicción no coincide con la clase real**: son los errores del modelo.

> 🔎 **Verificación.** Comprobemos que `predict` es efectivamente "probabilidad de clase 1 mayor o igual a 0,5". Si la expresión da `True`, coinciden en todos los registros.

In [ ]:
# Convertimos la probabilidad en clase aplicando el umbral 0,5 "a mano"
clase_manual = (y_test_proba[:, 1] >= 0.5).astype(int)    # True/False -> 1/0

print("¿predict() coincide con aplicar el umbral 0,5 a mano?", np.array_equal(clase_manual, y_test_pred))

## 4.7 Visualizar la curva y la frontera de decisión

Dibujamos los datos de prueba junto con la **curva sigmoide aprendida**, usando nuestra función `sigmoid_function` con los parámetros `w0` y `w1` que encontró el modelo.

También calculamos la **frontera de decisión**: el valor de `x` donde la probabilidad es exactamente 0,5. Eso ocurre cuando `z = 0`, es decir:

$$ w_0 + w_1 x = 0 \quad\Longrightarrow\quad x = -\frac{w_0}{w_1} $$

A la izquierda de ese valor el modelo predice clase 0; a la derecha, clase 1.

In [ ]:
frontera = -w0 / w1                                     # x donde la probabilidad vale 0,5
print(f"Frontera de decisión: x = {frontera:.3f}")

x_plot = np.linspace(-7, 12, 200)

plt.figure(figsize=(12, 6))
plt.scatter(X_test, y_test, c=y_test, alpha=0.5, lw=0.5, edgecolor="grey", cmap=cm.coolwarm, label="Datos de prueba")
plt.plot(x_plot, sigmoid_function(x_plot, w0, w1), color="green", linewidth=2.5, label="Curva sigmoide aprendida")
plt.axhline(0.5, color="gray", linestyle="--", label="Umbral = 0,5")
plt.axvline(frontera, color="black", linestyle="--", label=f"Frontera de decisión (x = {frontera:.2f})")
plt.xlabel("x")
plt.ylabel("Probabilidad de clase 1")
plt.title("Regresión logística con un atributo")
plt.legend()
plt.show()

La curva verde sube de 0 a 1 en la zona donde las clases se mezclan. La línea vertical negra marca la **frontera**: todo lo que quede a su izquierda se clasifica como clase 0 (azul) y a su derecha, como clase 1 (rojo).

## 4.8 Comprobar que la curva coincide con `predict_proba`

Como la función `sigmoid_function` implementa exactamente la misma fórmula que usa el modelo, tiene que dar **los mismos resultados** que `predict_proba`. Lo verificamos.

In [ ]:
proba_manual = sigmoid_function(X_test.ravel(), w0, w1)        # nuestra función
proba_modelo = clf.predict_proba(X_test)[:, 1]                 # la del modelo (columna de clase 1)

print("¿Coinciden las probabilidades?", np.allclose(proba_manual, proba_modelo))

Coinciden (`np.allclose` compara dos arreglos admitiendo diferencias minúsculas por redondeo). Es decir: **no hay magia**; el modelo es simplemente una función sigmoide con los `w0` y `w1` que aprendió.

---
# 5. Evaluación de un clasificador

En regresión medíamos *qué tan lejos* estaba la predicción del valor real (MAE, RMSE). En clasificación la pregunta es otra: **¿acertó o no acertó la clase?** Por eso se usan métricas distintas.

## 5.1 Accuracy (exactitud)

$$ Accuracy = \frac{\text{cantidad de predicciones correctas}}{\text{cantidad total de predicciones}} $$

Es el **porcentaje de aciertos**. Si el modelo clasificó bien 98 de 100 registros, su accuracy es 0,98 (98 %).

- ✅ Muy **fácil de entender** y de explicar.
- ❌ **Engañosa cuando las clases están desbalanceadas** (lo vamos a demostrar en la sección 5.4).
- ❌ **No distingue el tipo de error**: no es lo mismo equivocarse diciendo "sano" a alguien enfermo que decir "enfermo" a alguien sano.

Veamos el accuracy en entrenamiento y en prueba:

In [ ]:
from sklearn.metrics import accuracy_score

y_train_pred = clf.predict(X_train)

acc_train = accuracy_score(y_train, y_train_pred)
acc_test = accuracy_score(y_test, y_test_pred)

print(f"Accuracy en entrenamiento: {acc_train:.4f}  ({acc_train*100:.1f} %)")
print(f"Accuracy en prueba:        {acc_test:.4f}  ({acc_test*100:.1f} %)")

Los dos valores son muy altos y **parecidos entre sí**, señal de que el modelo **generaliza bien** (no memorizó los datos de entrenamiento). Pero un solo número no cuenta toda la historia. Veamos qué tipo de errores comete.

## 5.2 Matriz de confusión

La **matriz de confusión** es una tabla que cuenta **cuántos registros de cada clase real fueron clasificados en cada clase predicha**. Para clasificación binaria tiene cuatro casillas:

| | **Predicho: 0 (negativo)** | **Predicho: 1 (positivo)** |
|---|---|---|
| **Real: 0 (negativo)** | **VN** — Verdadero Negativo ✅ | **FP** — Falso Positivo ❌ |
| **Real: 1 (positivo)** | **FN** — Falso Negativo ❌ | **VP** — Verdadero Positivo ✅ |

- **VP (Verdadero Positivo):** era positivo y el modelo dijo positivo. *Acierto.*
- **VN (Verdadero Negativo):** era negativo y el modelo dijo negativo. *Acierto.*
- **FP (Falso Positivo):** era negativo pero el modelo dijo positivo. *Falsa alarma.*
- **FN (Falso Negativo):** era positivo pero el modelo dijo negativo. *Caso que se nos escapó.*

> 🏥 **Ejemplo con un test médico (positivo = enfermo):**
> - **FP:** el test dice "enfermo" pero la persona está sana → susto y estudios innecesarios.
> - **FN:** el test dice "sano" pero la persona está enferma → **no recibe el tratamiento a tiempo**. Suele ser el error más grave.
>
> Los dos errores **no cuestan lo mismo**, y la matriz de confusión permite verlos por separado.

*(Detalles de código: `confusion_matrix` calcula la tabla. `sns.heatmap(..., annot=True, fmt="d")` la dibuja como un mapa de calor mostrando los números enteros dentro de cada casilla.)*

In [ ]:
from sklearn.metrics import confusion_matrix

cm_test = confusion_matrix(y_test, y_test_pred)       # filas = clase real, columnas = clase predicha
print(cm_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicho 0", "Predicho 1"],
            yticklabels=["Real 0", "Real 1"])
plt.title("Matriz de confusión (conjunto de prueba)")
plt.show()

# .ravel() "estira" la matriz de 2x2 en una lista de 4 valores, en este orden: VN, FP, FN, VP
VN, FP, FN, VP = cm_test.ravel()
print(f"\nVN = {VN}, FP = {FP}, FN = {FN}, VP = {VP}")

Los números grandes están en la **diagonal** (de arriba a la izquierda a abajo a la derecha): son los aciertos. Los números chicos fuera de la diagonal son los errores (FP y FN). Un buen clasificador tiene la diagonal "cargada" y el resto cerca de cero.

## 5.3 Precision, recall y F1

A partir de la matriz de confusión se calculan métricas que se enfocan en **cómo le va al modelo con la clase positiva**.

### Precision (precisión)

$$ Precision = \frac{VP}{VP + FP} $$

**Pregunta que responde:** *de todos los casos que el modelo marcó como positivos, ¿cuántos realmente lo eran?*

- Alta precision = **pocas falsas alarmas**.
- Importa cuando **un falso positivo es costoso**. Ejemplo: un filtro de spam; si marca como spam un correo importante, el usuario lo pierde.

### Recall (sensibilidad / exhaustividad)

$$ Recall = \frac{VP}{VP + FN} $$

**Pregunta que responde:** *de todos los casos realmente positivos, ¿cuántos logró detectar el modelo?*

- Alto recall = **se le escapan pocos positivos**.
- Importa cuando **un falso negativo es costoso**. Ejemplo: detección de una enfermedad grave; es preferible algunas falsas alarmas antes que dejar sin detectar a un paciente enfermo.

### F1-score

$$ F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall} $$

Es un **promedio especial** (la *media armónica*) entre precision y recall: resume ambos en un solo número. Es alto **solo si las dos son altas**. Útil cuando querés un equilibrio entre ambas, especialmente con clases desbalanceadas.

| Métrica | Pregunta | Ventaja | Desventaja |
|---|---|---|---|
| **Accuracy** | ¿Qué porcentaje acertó? | Simple de entender | Engaña con clases desbalanceadas |
| **Precision** | De los que dije "positivo", ¿cuántos lo eran? | Controla las falsas alarmas | Ignora los positivos que no detectó |
| **Recall** | De los positivos reales, ¿cuántos detecté? | Controla los casos que se escapan | Un modelo que dice "positivo" a todo tiene recall perfecto |
| **F1** | ¿Hay equilibrio entre precision y recall? | Un solo número que combina ambas | Menos intuitiva; trata ambos errores como igual de graves |

> ⚖️ **Hay un compromiso (*trade-off*) entre precision y recall:** mejorar una suele empeorar la otra. Lo vamos a ver con el umbral en la sección 5.5.

Calculamos las tres métricas, primero **a mano** con la matriz de confusión y luego con **Scikit-Learn**.

In [ ]:
# --- A mano, usando VP, FP y FN de la matriz de confusión ---
precision_manual = VP / (VP + FP)
recall_manual = VP / (VP + FN)
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)

print("--- A mano ---")
print(f"Precision: {precision_manual:.4f}")
print(f"Recall:    {recall_manual:.4f}")
print(f"F1:        {f1_manual:.4f}")

# --- Con Scikit-Learn ---
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n--- Con Scikit-Learn ---")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_test_pred):.4f}")

Ambos cálculos coinciden. También existe una función que **junta todo en un solo reporte**, `classification_report`, muy usada en la práctica. Muestra precision, recall y F1 **para cada clase** (la columna `support` es la cantidad de registros reales de cada clase).

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred, target_names=["Clase 0", "Clase 1"]))

## 5.4 La trampa del accuracy con clases desbalanceadas

Imaginá un test para detectar una enfermedad rara que afecta al **5 %** de la población. Un "modelo" que **siempre responda "sano"** acertaría en el 95 % de los casos... ¡y no detectaría **a ningún** enfermo! Tendría un accuracy excelente y sería completamente inútil.

Comprobémoslo. Creamos un dataset **desbalanceado** (95 % de clase 0 y 5 % de clase 1) con `make_classification`, y comparamos:

1. Un **"modelo tonto"** que siempre predice la clase 0 (la más frecuente). Lo creamos con `np.zeros_like(y_test)`: un arreglo de ceros del mismo tamaño que `y_test`.
2. Una **regresión logística** de verdad.

El parámetro `weights=[0.95, 0.05]` define la proporción de cada clase.

In [ ]:
from sklearn.datasets import make_classification

X_d, y_d = make_classification(n_samples=2000, n_features=2, n_informative=2, n_redundant=0,
                               n_clusters_per_class=1, weights=[0.95, 0.05],
                               class_sep=1.5, random_state=0)

print("Registros por clase [clase 0, clase 1]:", np.bincount(y_d))

Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_d, y_d, test_size=0.3, random_state=42, stratify=y_d)

# 1) Modelo "tonto": siempre predice 0
pred_tonto = np.zeros_like(yd_test)

# 2) Regresión logística real
clf_d = LogisticRegression().fit(Xd_train, yd_train)
pred_real = clf_d.predict(Xd_test)

comparacion = pd.DataFrame({
    "Modelo tonto (siempre 0)": {"Accuracy": accuracy_score(yd_test, pred_tonto),
                                 "Recall (clase 1)": recall_score(yd_test, pred_tonto),
                                 "F1 (clase 1)": f1_score(yd_test, pred_tonto)},
    "Regresión logística":      {"Accuracy": accuracy_score(yd_test, pred_real),
                                 "Recall (clase 1)": recall_score(yd_test, pred_real),
                                 "F1 (clase 1)": f1_score(yd_test, pred_real)},
}).T

comparacion.round(3)

**El modelo tonto tiene un accuracy de ~94 % (casi el porcentaje de la clase mayoritaria), pero un recall de 0**: no detectó ningún caso positivo. La regresión logística tiene un accuracy más alto (~99 %) y, sobre todo, **sí detecta** la gran mayoría de los positivos (recall de ~0,8). Fijate que el accuracy del modelo tonto *parecía* excelente: justamente esa es la trampa.

> **Moraleja:** cuando las clases están desbalanceadas, **no te quedes con el accuracy**. Mirá la matriz de confusión, el recall, la precision y el F1 de la **clase que te interesa detectar**.

## 5.5 El umbral de decisión: el compromiso entre precision y recall

Hasta ahora usamos el umbral por defecto de **0,5**. Pero **podemos cambiarlo**, porque el modelo nos da las probabilidades (`predict_proba`):

- **Umbral más bajo** (por ejemplo, 0,3): el modelo se anima a decir "positivo" con menos evidencia → **más recall** (detecta más positivos) pero **menos precision** (más falsas alarmas).
- **Umbral más alto** (por ejemplo, 0,7): el modelo exige más evidencia → **más precision** pero **menos recall** (se le escapan más positivos).

Lo probamos con un dataset donde las clases **se superponen bastante** (para que haya errores que mostrar). Fijate que no hay que re-entrenar nada: solo cambiamos **cómo convertimos la probabilidad en clase**.

In [ ]:
# Dataset con bastante superposición entre clases (centros cercanos y mucha dispersión)
X_s, y_s = make_blobs(n_samples=1000, centers=[[0], [2.5]], cluster_std=1.8, random_state=3)
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_s, y_s, test_size=0.3, random_state=42, stratify=y_s)

clf_s = LogisticRegression().fit(Xs_train, ys_train)
proba_s = clf_s.predict_proba(Xs_test)[:, 1]            # probabilidad de clase 1

filas = []
for umbral in [0.2, 0.35, 0.5, 0.65, 0.8]:
    pred_umbral = (proba_s >= umbral).astype(int)       # aplicamos el umbral elegido
    filas.append({
        "Umbral": umbral,
        "Precision": precision_score(ys_test, pred_umbral),
        "Recall": recall_score(ys_test, pred_umbral),
        "F1": f1_score(ys_test, pred_umbral),
        "Accuracy": accuracy_score(ys_test, pred_umbral),
    })

pd.DataFrame(filas).round(3)

Mirá cómo se comportan las columnas al subir el umbral:

- El **recall baja** (el modelo detecta menos positivos).
- La **precision sube** (los que marca como positivos son más confiables).

**¿Qué umbral elegir? Depende del problema:**

| Situación | Error más grave | Conviene... |
|---|---|---|
| Detección de una enfermedad grave | Falso negativo (no detectar al enfermo) | Umbral **bajo** (priorizar recall) |
| Filtro de spam | Falso positivo (perder un correo importante) | Umbral **alto** (priorizar precision) |
| Sin preferencia clara | Ambos similares | Umbral cercano a 0,5 (priorizar F1) |

> Esta posibilidad de **ajustar el umbral** es una gran ventaja de la regresión logística frente a modelos que solo devuelven la clase.

---
# 6. Regresión logística con varios atributos

Con **un solo atributo** la frontera es un **punto** en una recta. Con **dos atributos**, la frontera es una **línea** en el plano. Con tres, un plano, y así sucesivamente. Matemáticamente es lo mismo: solo se suman más términos a la parte lineal:

$$ z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 \qquad\Longrightarrow\qquad P(\text{clase 1}) = \frac{1}{1 + e^{-z}} $$

La frontera de decisión es el conjunto de puntos donde la probabilidad vale 0,5, es decir, donde $z = 0$:

$$ \beta_0 + \beta_1 x_1 + \beta_2 x_2 = 0 $$

Esa ecuación es **una recta**. Por eso se dice que la regresión logística es un **clasificador lineal**: separa las clases con una línea recta (o un plano, o un hiperplano).

## 6.1 Crear y graficar los datos

Usamos otra vez `make_blobs`, pero ahora con `n_features=2` (dos atributos). Para controlar bien dónde caen las nubes, le indicamos los **centros** manualmente con `centers=[[1, 1], [4, 3]]`: la clase 0 se centra en el punto (1, 1) y la clase 1 en (4, 3).

In [ ]:
X2, y2 = make_blobs(n_samples=600, centers=[[1, 1], [4, 3]], n_features=2,
                    cluster_std=1.3, random_state=7)

print("Forma de X2:", X2.shape, "-> 600 filas y 2 columnas (2 atributos)")
print("Registros por clase:", np.bincount(y2))

plt.figure(figsize=(9, 6))
plt.scatter(X2[:, 0], X2[:, 1], c=y2, alpha=0.6, edgecolor="grey", lw=0.5, cmap=cm.coolwarm)
plt.xlabel("Atributo 1 (x1)")
plt.ylabel("Atributo 2 (x2)")
plt.title("Dos clases con dos atributos")
plt.show()

Cada punto es un registro; su **posición** depende de sus dos atributos y su **color** indica la clase. Se ven dos nubes (azul abajo a la izquierda, roja arriba a la derecha) con una zona de superposición en el medio. La regresión logística va a buscar **la recta que mejor las separe**.

## 6.2 Dividir, entrenar y evaluar

Los pasos son **idénticos** a los del caso de un atributo. Solo cambia que `X` ahora tiene **2 columnas**, y por eso `coef_` tendrá **dos valores**: uno por atributo.

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)

clf2 = LogisticRegression()
clf2.fit(X2_train, y2_train)
y2_pred = clf2.predict(X2_test)

print("Coeficientes (b1, b2):", clf2.coef_[0].round(3))
print("Intercepto (b0):      ", clf2.intercept_[0].round(3))
print(f"\nAccuracy en prueba: {accuracy_score(y2_test, y2_pred):.4f}")
print(f"Accuracy en entrenamiento: {accuracy_score(y2_train, clf2.predict(X2_train)):.4f}")
print()
print(classification_report(y2_test, y2_pred, target_names=["Clase 0", "Clase 1"]))

## 6.3 Visualizar la frontera de decisión

Para **ver** cómo el modelo divide el plano, usamos una técnica muy común:

1. Armamos una **grilla** de puntos que cubra todo el plano (`np.meshgrid`).
2. Le pedimos al modelo que prediga la clase (o la probabilidad) en cada punto de la grilla.
3. **Coloreamos** el fondo según la predicción (`plt.contourf`) y encima dibujamos los datos reales.

Como vamos a repetir esto varias veces, definimos una **función** reutilizable, `graficar_frontera`. Dibuja el fondo con la **probabilidad de clase 1** (gradiente de azul a rojo) y una **línea negra** donde la probabilidad vale exactamente 0,5: esa es la frontera de decisión.

In [ ]:
def graficar_frontera(modelo, X, y, ax, titulo):
    # Dibuja los datos y las regiones de decisión de un clasificador con 2 atributos
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    # Grilla de 300 x 300 puntos que cubre todo el plano
    g1, g2 = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grilla = np.column_stack([g1.ravel(), g2.ravel()])

    # Probabilidad de clase 1 en cada punto de la grilla
    prob = modelo.predict_proba(grilla)[:, 1].reshape(g1.shape)

    ax.contourf(g1, g2, prob, levels=20, cmap=cm.coolwarm, alpha=0.4)      # fondo con gradiente
    ax.contour(g1, g2, prob, levels=[0.5], colors="black", linewidths=2)   # frontera (prob = 0,5)
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="grey", lw=0.5, cmap=cm.coolwarm, alpha=0.8)
    ax.set_title(titulo)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

fig, ax = plt.subplots(figsize=(9, 6))
graficar_frontera(clf2, X2_test, y2_test, ax, "Frontera de decisión (datos de prueba)")
plt.show()

La **línea negra** es la frontera de decisión: **recta**, como anticipamos. Todo lo que cae del lado azul se clasifica como clase 0 y lo del lado rojo, como clase 1. Los puntos "equivocados" son los que quedaron del lado del color contrario al suyo. El **degradé** muestra además la **confianza** del modelo: cerca de la frontera la probabilidad ronda 0,5 (zona de duda); lejos de ella, se acerca a 0 o a 1.

## 6.4 Un ejemplo más realista: ¿aprobará el examen?

Pasemos a un caso con sentido: predecir si un estudiante **aprueba (1) o no aprueba (0)** según sus **horas de estudio** y su **porcentaje de asistencia**. Generamos 400 estudiantes ficticios. La "regla oculta" (que el modelo no conoce) es:

$$ z = -8 + 0{,}7 \cdot horas + 0{,}06 \cdot asistencia \qquad P(aprobar) = \frac{1}{1+e^{-z}} $$

y para cada estudiante **sorteamos** si aprueba o no según esa probabilidad (`np.random.binomial`). Así hay azar: dos estudiantes con los mismos datos podrían terminar con resultados distintos, igual que en la vida real.

In [ ]:
np.random.seed(0)
n_est = 400

estudiantes = pd.DataFrame({
    "horas_estudio": np.random.uniform(0, 10, n_est),         # entre 0 y 10 horas semanales
    "asistencia_pct": np.random.uniform(40, 100, n_est),      # entre 40 % y 100 %
})

# Probabilidad real de aprobar (regla oculta) y resultado sorteado con esa probabilidad
z = -8 + 0.7 * estudiantes["horas_estudio"] + 0.06 * estudiantes["asistencia_pct"]
prob_aprobar = 1 / (1 + np.exp(-z))
estudiantes["aprobo"] = np.random.binomial(1, prob_aprobar)   # 1 = aprobó, 0 = no aprobó

print("Estudiantes que aprobaron (1) y que no (0):")
print(estudiantes["aprobo"].value_counts())
estudiantes.head()

In [ ]:
atributos = ["horas_estudio", "asistencia_pct"]
X_est = estudiantes[atributos]
y_est = estudiantes["aprobo"]

Xe_train, Xe_test, ye_train, ye_test = train_test_split(X_est, y_est, test_size=0.25, random_state=42, stratify=y_est)

clf_est = LogisticRegression().fit(Xe_train, ye_train)
ye_pred = clf_est.predict(Xe_test)

print(f"Accuracy en prueba: {accuracy_score(ye_test, ye_pred):.3f}")
print(f"Precision:          {precision_score(ye_test, ye_pred):.3f}")
print(f"Recall:             {recall_score(ye_test, ye_pred):.3f}")

# Coeficientes con el nombre de cada atributo
print("\nCoeficientes aprendidos:")
print(pd.Series(clf_est.coef_[0], index=atributos).round(3))
print(f"Intercepto: {clf_est.intercept_[0]:.3f}")

### ¿Cómo se interpretan los coeficientes?

Los coeficientes están en la escala del **logit** (el logaritmo de los *odds*). Ambos son **positivos**: más horas de estudio y más asistencia **aumentan** la probabilidad de aprobar. Pero un número como "0,6" en escala logit no es fácil de interpretar. Para traducirlo, aplicamos la exponencial $e^{\beta}$, que da el **factor por el cual se multiplican los odds** cuando ese atributo aumenta en 1 unidad (manteniendo los demás constantes):

- Si $e^{\beta} = 2$ → los odds de aprobar **se duplican** por cada unidad adicional.
- Si $e^{\beta} = 1$ → ese atributo **no influye**.
- Si $e^{\beta} < 1$ → ese atributo **reduce** los odds (el coeficiente era negativo).

In [ ]:
odds_ratio = np.exp(clf_est.coef_[0])          # e elevado a cada coeficiente

print("Factor por el que se multiplican los odds de aprobar:")
print(pd.Series(odds_ratio, index=["por cada hora extra de estudio", "por cada 1 punto % de asistencia"]).round(3))

**Interpretación:** cada **hora extra de estudio** multiplica los odds de aprobar por aproximadamente **1,9** (casi los duplica); cada **punto porcentual más de asistencia** los multiplica por cerca de **1,05** (un 5 % más). Los valores se acercan a los reales de la regla oculta ($e^{0,7} \approx 2{,}01$ y $e^{0,06} \approx 1{,}06$); no coinciden exactamente porque el modelo aprende de una muestra con azar. Esta capacidad de **explicar** el efecto de cada atributo es una de las grandes virtudes del modelo.

### Predecir para un estudiante nuevo

Le pasamos al modelo un DataFrame con **las mismas columnas** con las que se entrenó. Usamos `predict_proba` para ver la **probabilidad** además de la clase.

In [ ]:
estudiante_nuevo = pd.DataFrame({"horas_estudio": [6], "asistencia_pct": [80]})

proba = clf_est.predict_proba(estudiante_nuevo)[0]          # [P(no aprobar), P(aprobar)]
clase = clf_est.predict(estudiante_nuevo)[0]

print(f"Probabilidad de NO aprobar: {proba[0]:.1%}")
print(f"Probabilidad de aprobar:    {proba[1]:.1%}")
print(f"Predicción: {'APRUEBA' if clase == 1 else 'NO APRUEBA'}")
print(f"\nSegún la regla oculta, la probabilidad real era: {1 / (1 + np.exp(-(-8 + 0.7*6 + 0.06*80))):.1%}")

El modelo no solo dice "aprueba" o "no aprueba": entrega **qué tan seguro está**. Eso es muy valioso en la práctica (por ejemplo, para decidir a qué estudiantes ofrecerles tutorías: a los que tienen una probabilidad intermedia o baja).

---
# 7. Variantes y extensiones

La regresión logística básica es binaria y con frontera recta, pero puede extenderse. En esta sección vemos tres situaciones importantes.

| Variante | ¿Qué hace? | ¿Cuándo se usa? |
|---|---|---|
| **Binaria** | Dos clases (0 y 1) | Spam / no spam, enfermo / sano |
| **Multiclase** | Tres o más clases (*softmax*) | Reconocer dígitos (0–9), tipos de flor, categorías de producto |
| **Regularizada** (parámetro `C`) | Penaliza coeficientes muy grandes para evitar sobreajuste | Muchos atributos, pocos datos, o atributos correlacionados |
| **Con atributos polinómicos** | Agrega potencias (x², x₁·x₂...) para lograr fronteras **curvas** | Cuando las clases no se pueden separar con una recta |

## 7.1 Clasificación multiclase

Scikit-Learn extiende automáticamente la regresión logística a **más de dos clases**: usás **exactamente el mismo código**. Internamente, en lugar de la sigmoide usa una generalización llamada **softmax**, que reparte la probabilidad entre todas las clases **de modo que sumen 1**.

Probemos con **3 clases** y 2 atributos. Usamos `make_blobs` con `centers=3` y el mismo flujo de siempre.

In [ ]:
X3, y3 = make_blobs(n_samples=600, centers=[[0, 0], [4, 1], [2, 5]], cluster_std=1.2, random_state=5)
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.3, random_state=42, stratify=y3)

clf3 = LogisticRegression().fit(X3_train, y3_train)        # ¡el mismo código de siempre!
y3_pred = clf3.predict(X3_test)

print("Clases que conoce el modelo:", clf3.classes_)
print("Forma de coef_:", clf3.coef_.shape, "-> una fila de coeficientes por cada clase")
print(f"Accuracy en prueba: {accuracy_score(y3_test, y3_pred):.3f}")

# Con 3 clases, predict_proba devuelve 3 columnas que suman 1 en cada fila
proba3 = clf3.predict_proba(X3_test[:3])
print("\nProbabilidades de los primeros 3 registros de prueba:")
print(pd.DataFrame(proba3, columns=["P(clase 0)", "P(clase 1)", "P(clase 2)"]).round(3))
print("Suma de cada fila:", proba3.sum(axis=1).round(3))

In [ ]:
# Visualizamos las regiones de decisión: cada color es una clase
x_min, x_max = X3[:, 0].min() - 1, X3[:, 0].max() + 1
y_min, y_max = X3[:, 1].min() - 1, X3[:, 1].max() + 1
g1, g2 = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
clases_grilla = clf3.predict(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)

plt.figure(figsize=(9, 6))
plt.contourf(g1, g2, clases_grilla, alpha=0.25, cmap="viridis", levels=[-0.5, 0.5, 1.5, 2.5])
plt.scatter(X3_test[:, 0], X3_test[:, 1], c=y3_test, cmap="viridis", edgecolor="grey", lw=0.5)
plt.title("Regresión logística con 3 clases")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

El plano queda dividido en **tres regiones**, una por clase, separadas por fronteras rectas. Con tres o más clases, la **matriz de confusión** pasa a ser de 3×3 (o N×N): sigue siendo cierto que la **diagonal** son los aciertos.

## 7.2 Regularización: el parámetro `C`

**Regularizar** significa agregarle al modelo una "penalización" para que **no use coeficientes demasiado grandes**. Esto lo hace más "conservador" y ayuda a **evitar el sobreajuste** (que el modelo memorice el ruido de los datos de entrenamiento).

En Scikit-Learn se controla con el parámetro **`C`**, que funciona **al revés** de lo que uno esperaría:

| Valor de `C` | Regularización | Efecto |
|---|---|---|
| **`C` chico** (0,01) | **Fuerte** | Coeficientes chicos, modelo más simple y "suave" |
| **`C` = 1** | Moderada (**valor por defecto**) | Equilibrio |
| **`C` grande** (100) | **Débil** | Coeficientes más libres, el modelo se ajusta más a los datos |

Veamos cómo cambian los coeficientes con distintos valores de `C` usando los datos de la sección 6.1:

In [ ]:
filas = []
for C in [0.001, 0.01, 1, 100]:
    modelo_C = LogisticRegression(C=C).fit(X2_train, y2_train)       # mismo modelo, distinto C
    filas.append({
        "C": C,
        "coef x1": modelo_C.coef_[0][0],
        "coef x2": modelo_C.coef_[0][1],
        "Accuracy (prueba)": accuracy_score(y2_test, modelo_C.predict(X2_test)),
    })

pd.DataFrame(filas).round(3)

Con `C` muy chico los **coeficientes son mucho menores** (el modelo está "frenado"), y a medida que `C` aumenta se acercan a los que el modelo elegiría libremente. En este dataset sencillo la accuracy casi no cambia, pero en problemas con muchos atributos y pocos datos, **ajustar `C`** puede marcar una gran diferencia.

> 📝 **Nota importante:** la regularización funciona mejor cuando los atributos están en **escalas similares**. Si un atributo va de 0 a 1 y otro de 0 a 1.000.000, conviene **estandarizarlos** antes (con `StandardScaler` de Scikit-Learn). Es un paso habitual en proyectos reales.

## 7.3 Cuando una recta no alcanza: fronteras curvas

La regresión logística básica solo sabe separar con **rectas**. ¿Qué pasa si las clases están dispuestas de forma que **ninguna recta** las separa? Por ejemplo, una clase "adentro" y otra "alrededor", como dos anillos concéntricos. Lo probamos con `make_circles`, y comparamos:

1. Regresión logística **común**.
2. Regresión logística con **atributos polinómicos** (`PolynomialFeatures(degree=2)`): agrega automáticamente los términos $x_1^2$, $x_2^2$ y $x_1 x_2$, lo que permite fronteras **curvas** (círculos, elipses). Usamos `make_pipeline` para encadenar los dos pasos en un solo modelo.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

Xc, yc = make_circles(n_samples=600, factor=0.4, noise=0.1, random_state=0)   # dos anillos concéntricos
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.3, random_state=42, stratify=yc)

# Modelo 1: regresión logística común (frontera recta)
modelo_recto = LogisticRegression().fit(Xc_train, yc_train)

# Modelo 2: regresión logística con atributos polinómicos de grado 2 (frontera curva)
modelo_curvo = make_pipeline(PolynomialFeatures(degree=2), LogisticRegression()).fit(Xc_train, yc_train)

print(f"Accuracy con frontera recta:  {accuracy_score(yc_test, modelo_recto.predict(Xc_test)):.3f}")
print(f"Accuracy con frontera curva:  {accuracy_score(yc_test, modelo_curvo.predict(Xc_test)):.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
graficar_frontera(modelo_recto, Xc_test, yc_test, ejes[0], "Regresión logística común")
graficar_frontera(modelo_curvo, Xc_test, yc_test, ejes[1], "Con atributos polinómicos (grado 2)")
plt.tight_layout()
plt.show()

La regresión logística común **no puede** separar los anillos (su accuracy queda por debajo del 50 %, peor que tirar una moneda), porque ninguna recta deja "un anillo de cada lado". Con atributos polinómicos, la frontera se vuelve una **curva cerrada** y el modelo logra casi un 100 %.

> **Moraleja:** igual que en la regresión lineal, **graficá siempre los datos** antes de elegir el modelo. Si las clases no se separan con una recta, hay que transformar los atributos o usar otro algoritmo (árboles de decisión, SVM con kernel, redes neuronales...).

---
# 8. Ventajas, desventajas y cuándo usar regresión logística

## 8.1 Ventajas ✅

| Ventaja | Explicación |
|---|---|
| **Simple y rápida** | Se entrena en una fracción de segundo, incluso con muchos datos |
| **Entrega probabilidades** | No solo dice "clase 1": dice **con qué confianza**. Permite ajustar el umbral según el problema |
| **Interpretable** | Los coeficientes (y sus *odds ratio*) explican cómo influye cada atributo; se puede justificar una decisión ante personas no técnicas |
| **Buen punto de partida (*baseline*)** | Es el primer modelo que se prueba en muchos problemas de clasificación |
| **Funciona bien con pocos datos** | No necesita enormes cantidades de datos para dar resultados razonables |
| **Admite regularización** | El parámetro `C` ayuda a controlar el sobreajuste |
| **Extensible** | Se adapta a multiclase y, con atributos polinómicos, a fronteras curvas |

## 8.2 Desventajas ❌

| Desventaja | Explicación |
|---|---|
| **Frontera lineal** | Si las clases no se pueden separar con una recta (o un plano), el modelo falla (ejemplo de los anillos) |
| **Sensible a atributos irrelevantes o correlacionados** | Atributos casi duplicados vuelven inestables a los coeficientes (*multicolinealidad*) |
| **Sensible a outliers** | Valores extremos pueden desplazar la frontera |
| **Requiere atributos numéricos** | Las categorías (por ejemplo, "ciudad") hay que convertirlas antes en números (*encoding*) |
| **Mejora con atributos en escalas similares** | Sobre todo si se usa regularización |
| **Se resiente con clases muy desbalanceadas** | Tiende a favorecer la clase mayoritaria; requiere elegir bien las métricas y el umbral |
| **Menos potente que modelos complejos** | En problemas con relaciones muy complicadas, un árbol, un *random forest* o una red neuronal pueden rendir mejor |

## 8.3 Supuestos principales

1. **La variable a predecir es categórica** (en el caso básico, binaria).
2. **Independencia:** los registros no deben depender unos de otros.
3. **Relación lineal entre los atributos y el logit** (el logaritmo de los odds): es la forma que asume el modelo.
4. **Poca multicolinealidad:** los atributos no deberían ser casi copias unos de otros.
5. **Cantidad suficiente de registros de cada clase**, especialmente de la clase menos frecuente.

> No hace falta que se cumplan a la perfección: son una guía. Cuanto más se violen, menos confiable será el modelo.

## 8.4 ¿Cuándo conviene usarla?

| Usala cuando... | Considerá otra alternativa cuando... |
|---|---|
| Querés predecir una **categoría** (sobre todo binaria) | Querés predecir un **número** → regresión lineal u otros modelos de regresión |
| Necesitás **probabilidades**, no solo la clase | Las clases están separadas por **fronteras muy curvas o complejas** → árboles, SVM con kernel, redes neuronales |
| Necesitás **explicar** el modelo (medicina, finanzas, decisiones que afectan a personas) | Solo importa la máxima precisión y no la explicación |
| Tenés **pocos datos** o querés un modelo **rápido** como punto de partida | Tenés enormes volúmenes de datos con relaciones muy complejas |
| El gráfico sugiere que las clases se pueden **separar razonablemente con una línea** | Los atributos son imágenes, sonido o texto "crudo" (suelen requerir modelos más especializados) |

---
# 9. Ejercicios para practicar ✍️

Es momento de que lo hagas vos. Cada ejercicio tiene una **pista**. Escribí tu código en las celdas vacías (donde dice `# Tu código acá`).

---
### Ejercicio 1: tu primer clasificador desde cero

Generá un dataset con `make_blobs` de **400 registros**, **1 atributo**, **2 clases**, `cluster_std=2` y `random_state=0`. Luego:

1. Mostrá cuántos registros hay de cada clase.
2. Dividí en entrenamiento (75 %) y prueba (25 %), **estratificando** por clase.
3. Entrená una regresión logística.
4. Calculá la **frontera de decisión** (`x = -w0 / w1`).
5. Calculá el accuracy en prueba y mostrá la matriz de confusión.
6. Graficá los datos y la curva sigmoide aprendida, marcando la frontera.

> 💡 **Pista:** repetí los pasos de las secciones 4.2 a 4.7. Podés reutilizar la función `sigmoid_function` y recordá que `w0 = clf.intercept_[0]` y `w1 = clf.coef_[0][0]`.

In [ ]:
# Tu código acá

---
### Ejercicio 2: elegir el umbral

Imaginá que el modelo del ejercicio anterior (o el de la sección 5.5, `clf_s` con `proba_s`) detecta una **enfermedad grave**, donde **un falso negativo es mucho peor que un falso positivo**.

1. Calculá la precision, el recall y el F1 para umbrales de **0.1, 0.3, 0.5 y 0.7**.
2. Mostrá la matriz de confusión para el umbral 0.5 y para el 0.1.
3. ¿Qué umbral elegirías para este problema? Justificá en una o dos oraciones.

> 💡 **Pista:** usá `predict_proba(X_test)[:, 1]` y convertí a clases con `(proba >= umbral).astype(int)`. Si el falso negativo es lo más grave, buscás **maximizar el recall**.

In [ ]:
# Tu código acá

---
### Ejercicio 3: la trampa del accuracy

Generá un dataset **muy desbalanceado** con `make_classification(n_samples=3000, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, weights=[0.97, 0.03], class_sep=1.2, random_state=1)`.

1. Dividí en entrenamiento y prueba (70 % / 30 %, estratificado).
2. Calculá el accuracy de un "modelo tonto" que siempre predice la clase 0.
3. Entrená una regresión logística y calculá su accuracy, su recall y su F1 para la clase 1.
4. Mostrá la matriz de confusión del modelo. ¿Cuántos positivos reales logró detectar?
5. Probá bajar el umbral a **0.2**. ¿Qué pasa con el recall y con la precision?

> 💡 **Pista:** `np.zeros_like(y_test)` crea las predicciones del modelo tonto. Recordá que el recall se calcula **sobre la clase positiva (1)**.

In [ ]:
# Tu código acá

---
### Ejercicio 4 (desafío): clasificación multiclase

Generá un dataset con `make_blobs(n_samples=900, centers=[[0, 0], [5, 0], [2.5, 4]], cluster_std=1.5, random_state=2)` (3 clases, 2 atributos). Luego:

1. Dividí en entrenamiento y prueba (70 % / 30 %, estratificado) y entrená una regresión logística.
2. Calculá el accuracy y mostrá la **matriz de confusión de 3×3**. ¿Entre qué clases se producen más confusiones?
3. Imprimí el `classification_report`.
4. Elegí un punto nuevo, por ejemplo `[[2.5, 1.5]]`, y mostrá las **probabilidades de las tres clases**. ¿Suman 1?
5. Graficá las regiones de decisión (podés basarte en la celda de la sección 7.1).

> 💡 **Pista:** `confusion_matrix` funciona igual con más de 2 clases; en `sns.heatmap` ajustá `xticklabels` e `yticklabels` a tres etiquetas. Con 3 clases, `predict_proba` devuelve 3 columnas.

In [ ]:
# Tu código acá

---
# 10. Resumen y glosario

## 10.1 Lo que aprendiste

1. **Clasificación:** predecir una **categoría** (no un número). La regresión logística es un clasificador, a pesar de su nombre.
2. **Función sigmoide:** convierte cualquier número en un valor entre 0 y 1, que se interpreta como **probabilidad** de la clase 1.
3. **Cómo decide:** calcula $P(\text{clase 1}) = \frac{1}{1+e^{-(\beta_0 + \beta_1 x_1 + \dots)}}$ y aplica un **umbral** (normalmente 0,5).
4. **Parámetros:** `β₀` desplaza la curva y `β₁` controla su inclinación. La **frontera de decisión** (con un atributo) está en `x = -β₀/β₁`.
5. **Flujo en Scikit-Learn:** instanciar → `fit` → `predict` / `predict_proba`.
6. **Evaluación:** accuracy, **matriz de confusión** (VP, VN, FP, FN), **precision**, **recall** y **F1**. Con clases desbalanceadas, el accuracy engaña.
7. **Umbral ajustable:** subir o bajar el umbral intercambia precision por recall.
8. **Extensiones:** multiclase, regularización (`C`) y atributos polinómicos para fronteras curvas.

## 10.2 Código esencial para recordar

```python
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

clf = LogisticRegression()                 # 1. instanciar
clf.fit(X_train, y_train)                  # 2. entrenar
y_pred = clf.predict(X_test)               # 3. predecir clases
y_proba = clf.predict_proba(X_test)[:, 1]  #    o probabilidades de clase 1

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

## 10.3 Glosario

| Término | Definición |
|---|---|
| **Clasificación** | Problema donde se predice una categoría |
| **Clase** | Cada categoría posible (0, 1, ...) |
| **Clase positiva** | La clase (1) que nos interesa detectar |
| **Sigmoide** | Función en forma de "S" que convierte cualquier número en un valor entre 0 y 1 |
| **Probabilidad** | Número entre 0 y 1 que expresa qué tan probable es un suceso |
| **Umbral** | Probabilidad mínima a partir de la cual se predice la clase 1 (por defecto 0,5) |
| **Frontera de decisión** | Borde que separa las regiones donde el modelo predice cada clase |
| **Odds** | Cociente p / (1 − p): chances a favor de un suceso |
| **Logit** | Logaritmo de los odds; es lineal en los atributos |
| **Accuracy** | Proporción de predicciones correctas |
| **Matriz de confusión** | Tabla que cuenta aciertos y errores de cada clase |
| **VP / VN / FP / FN** | Verdadero positivo, verdadero negativo, falso positivo, falso negativo |
| **Precision** | De los que el modelo marcó como positivos, cuántos lo eran |
| **Recall (sensibilidad)** | De los positivos reales, cuántos detectó el modelo |
| **F1** | Media armónica de precision y recall |
| **Dataset desbalanceado** | Dataset donde una clase es mucho más frecuente que otra |
| **Estratificar** | Dividir los datos manteniendo la proporción de cada clase |
| **Regularización (`C`)** | Penalización que limita el tamaño de los coeficientes para evitar el sobreajuste |
| **Softmax** | Generalización de la sigmoide para más de dos clases |

## 10.4 ¿Y ahora qué?

- **Profundizar en métricas:** curva **ROC** y **AUC**, y curva **precision-recall**, que evalúan al modelo para *todos* los umbrales a la vez.
- **Otros clasificadores:** árboles de decisión, *random forest*, k-vecinos más cercanos (KNN), SVM.
- **Validación cruzada:** una forma más robusta de evaluar que una única división train/test.
- **Datos reales:** Scikit-Learn incluye datasets de práctica (por ejemplo `load_breast_cancer`, `load_iris`), y en Kaggle o UCI Machine Learning Repository encontrás muchos más.
- Explorá la [documentación de Scikit-Learn sobre regresión logística](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression).

**¡Felicitaciones por completar tu práctica de clasificación!** 🎉